# 06 — Deadline-aware larger training and final hybrid submission

Use **Python (ml-challenge-neural)**. **Run All** includes training, evaluation, full test prediction, validation and local packaging. Nothing is uploaded automatically.

**Submission deadline: 27 September 2026, 11:59 PM IST.** The notebook uses this fixed time, not a fresh countdown on each rerun. Start promptly. It targets 250,000 training references when at least 14 hours remain, otherwise 100,000. Preparation can stop sooner; actual counts are recorded. An existing experiment keeps its original plan on resume.

The encoder starts from your completed 05 weights and takes **one additional pass** at learning rate 1e-5. It uses batch 4, length 192, at most three positives and three explicit hard negatives per reference. This warm start differs from training a fresh encoder from scratch. Training stops with seven hours reserved; partial-but-valid weights may be evaluated. The classifier retains all retrieved training positives and up to eight hard negatives per reference. Tune/audit retain complete lexical candidate sets.

**This is a hybrid matching upgrade, not new dense retrieval.** All large text and embedding lookups are disk-backed. The default target is a plan, not a promised score or runtime. Keep at least 40 GiB free at first start. Original 04 outputs remain the fallback; a new package is written separately.

In [2]:
import os,sys,json,time,shutil,subprocess
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
os.environ.setdefault('POLARS_MAX_THREADS','8')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
ancestors=(Path.cwd(),*Path.cwd().parents)
SRC=next(p/r for p in ancestors for r in ('.','src','submission/code/business_entity_resolution/src','code/business_entity_resolution/src') if (p/r/'deadline_neural.py').exists()).resolve()
sys.path.insert(0,str(SRC))
import torch,polars as pl
from deadline_neural import *
from predict_submission import export_outputs,package_submission
assert torch.cuda.is_available(), 'Use the neural kernel with GPU access.'
CACHE=Path(os.environ['ER_CACHE_ROOT']).expanduser().resolve() if os.environ.get('ER_CACHE_ROOT') else next(p/'cache/normalized_v1' for p in ancestors if (p/'cache/normalized_v1').is_dir())
CANDIDATES=Path(os.environ.get('ER_CANDIDATE_ROOT',str(CACHE.parent/'candidates_v1'))).expanduser().resolve()
OLD=Path(os.environ.get('ER_NEURAL_ROOT',str(CACHE.parent/'neural_v1'))).expanduser().resolve()
WORK=Path(os.environ.get('ER_LARGE_ROOT',str(CACHE.parent/'neural_deadline_v1'))).expanduser().resolve()
WORK.mkdir(parents=True,exist_ok=True)
PROJECT=SRC.parents[3]
PACKAGE_ROOT=PROJECT/'submission_neural'
BASELINE_PACKAGE=SRC.parents[2]
DEADLINE=datetime(2026,9,27,23,59,tzinfo=ZoneInfo('Asia/Kolkata')).timestamp()
assert hours_left(DEADLINE)>0, 'The configured submission deadline has passed. Do not silently start a late experiment.'
TEAM_NAME='ML Specialists'
TEAM_MEMBERS='Rudra Prasad Dash, Dibyajyoti Parida, Sailen Sahoo'
PLAN_PATH=WORK/'experiment.json'
if PLAN_PATH.exists():
    plan=json.loads(PLAN_PATH.read_text())
    assert plan['deadline']==DEADLINE
else:
    assert shutil.disk_usage(WORK).free>40*2**30, 'Keep 40 GiB free before starting the large run.'
    plan={'deadline':DEADLINE,'train_target':250000 if hours_left(DEADLINE)>=14 else 100000,'holdout_per_role':10000,'created':time.time()}
    atomic_json(PLAN_PATH,plan)
(WORK/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
reference_manifest=cache_parts(CACHE,'test',1)[1]
DATA_ROOT=Path(os.environ.get('ER_DATA_ROOT',str(Path(reference_manifest['signature']['path']).parents[1]))).expanduser().resolve()
assert (DATA_ROOT/'test/test_source1.tsv').exists()
assert (OLD/'encoder/model.safetensors').exists() and (OLD/'lexical_plus_e5.json').exists()
print('GPU:',torch.cuda.get_device_name(0))
print('Hours until submission:',round(hours_left(DEADLINE),2))
print('Plan:',plan)
print('Work:',WORK,'Final package:',PACKAGE_ROOT)


GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Hours until submission: 4.9
Plan: {'deadline': 1790533740.0, 'train_target': 250000, 'holdout_per_role': 10000, 'created': 1790475576.3646584}
Work: /home/aline/Projects/ML-Challenge/cache/neural_deadline_v1 Final package: /home/aline/Projects/ML-Challenge/submission_neural


In [4]:
import importlib
import inference_fast_cache as fast

selected = json.loads((WORK / "selected.json").read_text())
audit_report = json.loads((WORK / "audit.json").read_text())

TEST_INDEX = CACHE.parent / "prediction_v1/test_search.sqlite"
TEST_FAST = CACHE.parent / "prediction_v1/test_fast"

importlib.reload(fast)
fast.install()
predict_all = fast.predict_all_timed
BASELINE_ONLY = False



Batched embedding-cache commits enabled. Existing prediction chunks and vectors will be reused.


## Prepare bounded-memory training data

Fresh tuning/audit references exclude both the 02 pilot and 05 references. Known positive targets shared with 05's training data are excluded from new holdouts; shared targets across new roles are purged with audit priority. Actual holdout sizes may therefore be below 10,000. This guards known identity overlap, not undisclosed duplicate identities absent from the labels.

Record stores checkpoint each normalized shard. Retrieval/features checkpoint every 500 references; all tune/audit candidates are retained. The training preparation budget is 1.5 hours and may reduce the actual training sample. Interruptions resume committed data. Changes to helper code, sources or configuration require a new `ER_LARGE_ROOT`.

If fewer than eight hours remain before preparation begins, the notebook skips the larger experiment and proceeds with the completed 05 model.

In [2]:
new_encoder=new_model=new_meta=None
roles=labels=None
audit_report={'status':'not_run','note':'Larger-run audit skipped to preserve submission time; see 05 validation separately.'}
if hours_left(DEADLINE)>=8 or (WORK/'prepared/manifest.json').exists():
    train_store=build_records(CACHE,'train',WORK/'train_records.sqlite')
    train_records=Records(train_store)
    try:
        roles,labels=choose_splits(CACHE,CANDIDATES/'pilot',OLD,WORK,plan['train_target'],plan['holdout_per_role'])
        display(roles.group_by('role','country_key').len().sort('role','country_key'))
        prepared=prepare_pairs(roles,labels,CANDIDATES/'train_search.sqlite',CANDIDATES/'train_fast',train_records,WORK,DEADLINE)
        print('Actual prepared training references:',prepared['training_references'])
    finally:train_records.close()
else:
    print('Time cutoff: using 05 hybrid without a new training experiment.')


train record store: 100,000/12,527,040, ETA 8.1 min
train record store: 200,000/12,527,040, ETA 8.5 min
train record store: 300,000/12,527,040, ETA 8.9 min
train record store: 400,000/12,527,040, ETA 9.1 min
train record store: 500,000/12,527,040, ETA 9.2 min
train record store: 600,000/12,527,040, ETA 9.3 min
train record store: 700,000/12,527,040, ETA 9.6 min
train record store: 800,000/12,527,040, ETA 9.7 min
train record store: 900,000/12,527,040, ETA 9.9 min
train record store: 1,000,000/12,527,040, ETA 10.1 min
train record store: 1,100,000/12,527,040, ETA 10.3 min
train record store: 1,200,000/12,527,040, ETA 10.7 min
train record store: 1,300,000/12,527,040, ETA 10.8 min
train record store: 1,400,000/12,527,040, ETA 11.0 min
train record store: 1,500,000/12,527,040, ETA 11.1 min
train record store: 1,600,000/12,527,040, ETA 11.4 min
train record store: 1,700,000/12,527,040, ETA 11.6 min
train record store: 1,800,000/12,527,040, ETA 11.7 min
train record store: 1,900,000/12,527,

role,country_key,len
str,str,u32
"""audit""","""india""",4166
"""audit""","""us""",5834
"""train""","""india""",100046
"""train""","""us""",149954
"""tune""","""india""",4022
"""tune""","""us""",5978


Prepared tune: 500; 75.3 refs/s; 15.9 h remaining
Prepared tune: 5,500; 109.0 refs/s; 15.9 h remaining
Prepared audit: 500; 117.1 refs/s; 15.9 h remaining
Prepared audit: 5,500; 121.7 refs/s; 15.9 h remaining
Prepared train: 500; 127.4 refs/s; 15.9 h remaining
Prepared train: 5,500; 137.7 refs/s; 15.9 h remaining
Prepared train: 10,500; 146.5 refs/s; 15.9 h remaining
Prepared train: 15,500; 153.5 refs/s; 15.9 h remaining
Prepared train: 20,500; 158.6 refs/s; 15.8 h remaining
Prepared train: 25,500; 162.9 refs/s; 15.8 h remaining
Prepared train: 30,500; 165.9 refs/s; 15.8 h remaining
Prepared train: 35,500; 169.2 refs/s; 15.8 h remaining
Prepared train: 40,500; 171.9 refs/s; 15.8 h remaining
Prepared train: 45,500; 174.4 refs/s; 15.8 h remaining
Prepared train: 50,500; 176.5 refs/s; 15.8 h remaining
Prepared train: 55,500; 178.6 refs/s; 15.8 h remaining
Prepared train: 60,500; 180.2 refs/s; 15.8 h remaining
Prepared train: 65,500; 182.1 refs/s; 15.8 h remaining
Prepared train: 70,500; 1

## Train one additional encoder pass, then fit XGBoost

The encoder saves optimizer/RNG checkpoints every 1,000 steps. It stops by **4:59 PM IST**, reserving seven hours. Disk-backed float32 embeddings are shared across training feature chunks. The classifier uses a quantized matrix with 128 histogram bins to reduce RAM/VRAM use.

A deadline cutoff during feature construction or before fitting switches to the completed 05 model. Unexpected errors are not silently hidden. The larger model is accepted only if it beats 05 on the new tuning set; audit labels are not used for selection.

In [3]:
if roles is not None:
    train_records=Records(WORK/'train_records.sqlite')
    try:
        new_encoder=train_large(train_records,WORK,OLD/'encoder',DEADLINE)
        if new_encoder is not None:
            try:
                feature_dir=embed_prepared(WORK,new_encoder,train_records,DEADLINE)
                new_model,new_meta=fit_hybrid(feature_dir,labels,roles,WORK,DEADLINE)
            except TimeoutError as error:
                print(error)
                new_model=new_meta=None
        selected=select_model(WORK,OLD,train_records,roles,labels,new_encoder,new_model,new_meta,DEADLINE)
        if hours_left(DEADLINE)>5:
            audit_report=audit_selected(WORK,selected,train_records,roles,labels,DEADLINE)
        print('Selected model:',json.dumps(selected,indent=2))
        print('Audit:',json.dumps(audit_report,indent=2))
    finally:train_records.close()
else:
    selected=select_model(WORK,OLD,None,None,None,None,None,None,DEADLINE)


/home/aline/Projects/ML-Challenge/.venv-neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoder step 100; 40.9 examples/s; loss 0.9817; 15.55 h remaining
Encoder step 200; 42.0 examples/s; loss 0.0075; 15.54 h remaining
Encoder step 300; 42.5 examples/s; loss 0.0001; 15.54 h remaining
Encoder step 400; 42.9 examples/s; loss 0.1428; 15.54 h remaining
Encoder step 500; 43.2 examples/s; loss 0.0015; 15.54 h remaining
Encoder step 600; 43.5 examples/s; loss 0.0129; 15.53 h remaining
Encoder step 700; 43.5 examples/s; loss 0.0021; 15.53 h remaining
Encoder step 800; 43.6 examples/s; loss 0.0067; 15.53 h remaining
Encoder step 900; 43.9 examples/s; loss 0.0159; 15.53 h remaining
Encoder step 1,000; 44.1 examples/s; loss 0.0031; 15.52 h remaining
Encoder step 1,100; 43.8 examples/s; loss 0.0015; 15.52 h remaining
Encoder step 1,200; 44.0 examples/s; loss 0.0246; 15.52 h remaining
Encoder step 1,300; 44.2 examples/s; loss 0.0862; 15.52 h remaining
Encoder step 1,400; 44.4 examples/s; loss 0.2175; 15.51 h remaining
Encoder step 1,500; 44.6 examples/s; loss 0.0132; 15.51 h remainin

## Build test text lookup and benchmark final inference

The completed 04 test retrieval index is reused unchanged. A separate record store retains the original-script inputs required by E5. Each encountered record is encoded once and cached on disk; every candidate is scored using the selected model.

The preflight separates lexical work from unique-record encoding so it does not extrapolate repeated encoding for all 82 million pairs. Its projection is still approximate. If it will not fit with a 45-minute validation/packaging buffer, the notebook uses the already validated 04 outputs instead of beginning an implausible late run.

In [4]:
BASELINE_ONLY=hours_left(DEADLINE)<3
if not BASELINE_ONLY:
    TEST_INDEX=CACHE.parent/'prediction_v1/test_search.sqlite'
    TEST_FAST=CACHE.parent/'prediction_v1/test_fast'
    assert TEST_INDEX.exists() and (TEST_FAST/'stats.json').exists(), 'Complete 04 first.'
    test_store=build_records(CACHE,'test',WORK/'test_records.sqlite')
    test_records=Records(test_store)
    try:
        preflight=test_preflight(CACHE,TEST_INDEX,TEST_FAST,test_records,selected,WORK)
        print(json.dumps(preflight,indent=2))
        remaining=max(0,reference_manifest['total_rows']-sum(v['rows'] for v in json.loads((WORK/'predictions/manifest.json').read_text())['parts'].values())) if (WORK/'predictions/manifest.json').exists() else reference_manifest['total_rows']
        projected=preflight['projected_scoring_hours']*remaining/reference_manifest['total_rows']
        BASELINE_ONLY=projected+.75>hours_left(DEADLINE)
        if BASELINE_ONLY:print('Deadline fallback: preserve the completed 04 submission.')
    finally:test_records.close()


test record store: 100,000/11,702,133, ETA 9.7 min
test record store: 200,000/11,702,133, ETA 10.1 min
test record store: 300,000/11,702,133, ETA 11.1 min
test record store: 400,000/11,702,133, ETA 11.8 min
test record store: 500,000/11,702,133, ETA 12.7 min
test record store: 600,000/11,702,133, ETA 13.0 min
test record store: 700,000/11,702,133, ETA 14.0 min
test record store: 800,000/11,702,133, ETA 14.0 min
test record store: 900,000/11,702,133, ETA 14.6 min
test record store: 1,000,000/11,702,133, ETA 14.4 min
test record store: 1,100,000/11,702,133, ETA 14.0 min
test record store: 1,200,000/11,702,133, ETA 14.5 min
test record store: 1,300,000/11,702,133, ETA 14.3 min
test record store: 1,400,000/11,702,133, ETA 14.0 min
test record store: 1,500,000/11,702,133, ETA 14.3 min
test record store: 1,600,000/11,702,133, ETA 14.0 min
test record store: 1,700,000/11,702,133, ETA 13.7 min
test record store: 1,732,544/11,702,133, ETA 13.7 min
test record store: 1,832,544/11,702,133, ETA 13

## Score every test reference and validate

This cell resumes hash-verified output parts after an interruption. The final TSVs contain all references, including empty predictions. Candidate lists are exactly the records scored. Full raw-ID validation checks membership, uniqueness and match-subset consistency.

Do not alter the selected model, token length or feature implementation during prediction. The new output directory remains separate from the baseline.

In [12]:
import importlib
import inference_fast_cache as fast

importlib.reload(fast)

precompute_status = fast.precompute_test_embeddings(
    WORK,
    selected,
    max_hours=2.0,
    reserve_hours=5.0,
)

fast.install()
predict_all = fast.predict_all_timed

Embedding timing: 4,096 IDs, 3,071 new; read 0.01s, encode/load 1.60s, write/commit 0.03s
Precompute: 5,402,624 records visited; 3,071 newly encoded this run; last chunk 1.7s (3,071 new); budget left 2.00h
Precompute: 5,439,488 records visited; 30,330 newly encoded this run; last chunk 1.8s (3,010 new); budget left 1.99h
Embedding timing: 4,096 IDs, 3,089 new; read 0.00s, encode/load 1.68s, write/commit 0.01s
Precompute: 5,480,448 records visited; 60,721 newly encoded this run; last chunk 2.3s (3,017 new); budget left 1.99h
Embedding timing: 4,096 IDs, 3,080 new; read 0.01s, encode/load 1.69s, write/commit 0.01s
Precompute: 5,521,408 records visited; 91,178 newly encoded this run; last chunk 2.2s (3,001 new); budget left 1.98h
Embedding timing: 4,096 IDs, 3,029 new; read 0.00s, encode/load 1.70s, write/commit 0.01s
Precompute: 5,562,368 records visited; 121,461 newly encoded this run; last chunk 1.7s (3,024 new); budget left 1.98h
Embedding timing: 4,096 IDs, 3,016 new; read 0.00s, enc

In [ ]:
if not BASELINE_ONLY:
    test_records=Records(WORK/'test_records.sqlite')
    try:prediction_dir=predict_all(CACHE,TEST_INDEX,TEST_FAST,test_records,selected,WORK)
    finally:test_records.close()
    export_outputs(prediction_dir,PACKAGE_ROOT/'output')
    validation_report=validate_outputs(PACKAGE_ROOT/'output/matching_results.tsv',PACKAGE_ROOT/'output/candidate_pairs.tsv',DATA_ROOT/'test',WORK/'validation')
    print(json.dumps(validation_report,indent=2))
else:
    print('Using existing baseline outputs:',BASELINE_PACKAGE/'output')


/home/aline/Projects/ML-Challenge/.venv-neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding timing: 11,974 IDs, 0 new; read 2.60s, encode/load 0.00s, write/commit 0.00s
Stage timing: source_read=0.10s, retrieval=1.66s, pair_frame=0.01s, target_read=3.29s, string_features=0.37s, feature_frame=0.01s, embeddings_cosine=2.62s, dmatrix=0.01s, xgboost=0.05s, output_build=0.01s, save=0.01s, total=8.14s
Stage timing: source_read=0.08s, retrieval=0.96s, pair_frame=0.01s, target_read=2.81s, string_features=0.32s, feature_frame=0.00s, embeddings_cosine=2.05s, dmatrix=0.00s, xgboost=0.00s, output_build=0.01s, save=0.01s, total=6.26s
Hybrid prediction 380,250/1,732,544; ETA 10.94 h
Stage timing: source_read=0.08s, retrieval=0.94s, pair_frame=0.01s, target_read=2.38s, string_features=0.34s, feature_frame=0.00s, embeddings_cosine=1.79s, dmatrix=0.01s, xgboost=0.00s, output_build=0.01s, save=0.01s, total=5.57s
Embedding timing: 12,111 IDs, 0 new; read 1.21s, encode/load 0.00s, write/commit 0.00s
Stage timing: source_read=0.06s, retrieval=0.94s, pair_frame=0.01s, target_read=1.38s, 

## Create a reviewable final package

Set `TEAM_NAME` and `TEAM_MEMBERS` in the setup cell (or assign them immediately here). Blank fields still stage the validated files and weights, but defer ZIP creation. Inspect the generated methodology. The ZIP has `output/`, `code/` and `Documentation_template.md` at its root.

The neural package includes encoder/tokenizer weights, XGBoost weights, inference source, and pinned environment definitions. Its `reproduce_neural.py` can regenerate outputs from normalized data without retraining. Nothing is uploaded automatically.

In [ ]:
if not BASELINE_ONLY:
    archive=package(WORK,PACKAGE_ROOT,SRC.parent,selected,audit_report,validation_report,TEAM_NAME,TEAM_MEMBERS)
    final_output=PACKAGE_ROOT/'output/matching_results.tsv'
else:
    baseline_work=CACHE.parent/'prediction_v1'
    baseline_model=CACHE.parent/'training_v1/model'
    report=json.loads((baseline_work/'validation/validation_report.json').read_text())
    parity=json.loads((baseline_model/'retrieval_parity.json').read_text())
    old_manifest=json.loads((baseline_work/'predictions/manifest.json').read_text())
    archive=package_submission(BASELINE_PACKAGE,baseline_model,report,parity,old_manifest,TEAM_NAME,TEAM_MEMBERS)
    final_output=BASELINE_PACKAGE/'output/matching_results.tsv'
print('Leaderboard TSV:',final_output)
print('Final archive:',archive or 'Set team fields and rerun this cell to create the ZIP.')
print('Review and upload before 27 September 2026, 11:59 PM IST.')
